## Business Problem

As an analyst at Netflix I've been tasked with researching what type of content to release on the platform.  

It's 2016 and the company is moving toward releasing our own content rather than distributing films and shows from the major production studios.

The executives would like a report on the probability of success. The market is competitive and we can only afford to spend $10B per year on movies. We need to be selective and have movies across multiple genres. We can't just create action movies. Our audience wants a variety of genres.

---

### The Model

The output of this analysis should result in a single probability of success score based on different genre and budget groupings. 

**Probability of success will be defined as...**
- Having ROI >= 100%. Movies need to make double their budget to cover production and distribution costs.
- Or if budget/revenue info isn't available, having an average movie rating that's in the top 25% of all movie data.

**It may look something like this:**

***Movie Success (ROI)*** = *movies[genre_list] + keywords[keywords_list] + movies[budget] + avg(ratings[rating] by director) + avg(ratings[ratings] by actor)*  

***ROI*** = *(movies[revenue] - movies[budget]) / movies[budget]*

If budget is zero and ratings avg exists use the model below. This 2nd scenario allows us to utilize all the data, even if budget/revenue are missing.  

 ***Avg Movie Rating avg(ratings[rating])*** = *movies[genre_list] + keywords[keywords_list] + avg(ratings[rating] by director) + avg(ratings[rating] by actor)*

To avoid overfitting the model and making it too complex, only predictors that have a significant impact on the ouctome will be included.

---

### Business Decision

Executives will be able to filter through different movie genres and budget options to select the movie attributes with the highest probability of success. Success can be measured by ROI or avg rating.

---
## Loading The Data <br>

The initial step in this is project is to load the required libraries and the datasets. 
 
The libraries required are...   
        - pathlib : a file handler that treats paths as objects rather than text strings.   
        - pandas : a library of functions that are used to manipulate dataframes.  
        - numpy : a library of math functions.  

Next, we define the root folder where the project folders and files will be saved.  

The project directory contains the following folders...  
        - The data folder : This folder contains raw data files.  
        - The output folder : This folder contains the final ABT and processed data files.    
        - The report folder : This folder contains html reports, graphs, and analysis.  



In [1]:
# This code loads libraries and sets directors for the project.
#  pathlib is a file handler. treats paths as objects not strings
# pandas brings in df functions
# numpy provides math functions.

#Checking my Python environment 
import sys; print(sys.executable); print(sys.version)

from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

#Set repository root to the current working directory. One step above readme location.
display(REPO_ROOT)

#Create directories for data, outputs, and reports if they don't exist.
DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

#set max column display to 100 rows for pandas dataframes.
pd.set_option("display.max_columns", 100)

/usr/bin/python3
3.13.5 (main, Aug 10 2026, 12:06:59) [GCC 14.2.0]


PosixPath('/workspaces/Project-Option-1-Matt-Flores')

---               
## Dataset Review

Now that the data is loaded we can review and profile each table loaded into the directory.  
There are 5 raw data files loaded into the data folder.
1) **movies** : This is the main df because the primary key, ***movieId***, in this df is the project's **unit of analysis**.  

2) **ratings** : This dataset contains almost 100k reviews throughout 1995 to 2016. This data will need to be aggregated to be merged with the ratings table. *Each row in this dataset represents a single user's rating for a movie. There are multiple ratings per movie.*  

3) **links** : Contains the imdbId & tmdbId associated with each movieId. It allows the ratings df to be joined with the credits and keywords tables without having to cross refence the movies dataset, which contains all 3 id types. *Each row in this dataset represents a movieId and related links. Each movieId should have 2 additional Ids.*  

4) **credits** : Lists the directors and top billed actors in each film. Maybe there is a correlation between revenue and directors/actors. *Each row in this dataset represents a movieID with associated directions/actors.*  

5) **keywords** : Contains The Movie Database ID along with a multi-valued field listing descriptive words for each movie. Genre is a broad categorization of a film. keywords df can provide more detail on themes, locations, movie plots, characters, etc. *Each row in this dataset represents a single movie.*

In [2]:
#This code loads the raw data into dataframes. And creates a column summary by table df.
# Load the datasets
movies = pd.read_csv(DATA_DIR / 'movies_metadata.csv')
ratings = pd.read_csv(DATA_DIR / 'ratings.csv', parse_dates=['rating_datetime'])
links = pd.read_csv(DATA_DIR / 'links.csv')
credits = pd.read_csv(DATA_DIR / 'credits_curated.csv')
keywords = pd.read_csv(DATA_DIR / 'keywords_curated.csv')

# Create a dictionary (keys paired with values) to hold the DataFrames for easier access. We can loop through tables, etc.
tables = {
    'movies': movies,
    'ratings': ratings,
    'links': links,
    'credits': credits,
    'keywords': keywords,
}

print("Datasets loaded into dataframes. Dictionary of dataframes called 'tables' created.")

Datasets loaded into dataframes. Dictionary of dataframes called 'tables' created.


---               
## Data Structure

The first step in data profiling is reviewing the data structure. Data stucture consists of rows and columns. To analyze data we must first understand what each row of the data represents and the attributes (columns) grouped with each row.

Instead of running shape() for each table I thought it would be best to aggregate each table's structure into summary table.  

I appended table name along with the associated columns into a single dataframe.  
I repeated the process a second time to gather the shape info from each table.

In [3]:
# Create a blank list called column_summary that we can populate.
column_summary = []

# Populate the list using a loop that appends table names and each column within that table.
for name, df in tables.items():
    column_summary.append({"table_name": name, **{f"col_{i+1}": col for i, col in enumerate(df.columns)}})

# Build a summary DataFrame that shows the table name and each column name in separate columns.
# Converted summary_data list into a pd.dataframe and set index equal to table name.
column_summary_df = pd.DataFrame(column_summary).set_index("table_name")

# Display the result
display(column_summary_df)

raw_data_shape = []
# Print out each df and it's shape. For is a loop function. "for every name in tables dictionary, print the shape variables"
for name, df in tables.items():
     raw_data_shape.append({"table_name": name, "rows": df.shape[0], "columns": df.shape[1]})

raw_data_shape_df = pd.DataFrame(raw_data_shape).set_index("table_name")

# Display the results
display(raw_data_shape_df)

,col_1,col_2,col_3,col_4,col_5,col_6,col_7,col_8,col_9,col_10,col_11,col_12,col_13,col_14,col_15,col_16,col_17,col_18,col_19,col_20,col_21
table_name,,,,,,,,,,,,,,,,,,,,,
movies,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
ratings,userId,movieId,rating,timestamp,rating_datetime,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
links,movieId,imdbId,tmdbId,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
credits,tmdbId,director,top_cast,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
keywords,tmdbId,keywords_list,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,rows,columns
table_name,,
movies,9082,21
ratings,99810,5
links,9125,3
credits,9082,3
keywords,9082,2


We can confirm a few facts:  
1) **Granularity Match**: The movies, credits, and keywords datasets contain 9,082 rows. The links df contains 43 more rows. It's possible the links table contains additional movieids that are not included in the other tables. There may also be duplicate ids in each table, but it seems these df's row level represents a single movie.  

2) **Ratings Shape**: The ratings df has almost 100k rows. The most popular movies might receive a large portion of these reviews. This will require further analysis.  

3) **Aggregation Required**: Since the ratings row level doesn't match the other tables any joining of the ratings table will require aggregation. We could examine the average rating per movie, distribution of review scores, count of reviews, etc.  

4) **Movie Sales Figures**: The movies dataset contains budget and revenue columns. These will be useful when calculating ROI per movie and sorting by revenue. One thing to consider is highly rated movies might not make the most revenue. Many classic movies such as "Fight Club" and "Donnie Darko" had a low or negative ROI.

---               
## Initial Inspection

This step is exploratory. We know the dataset structure, but it's helpful to see a snippet of each table. Let's loop through each table and output head(3) and tail(3)

In [4]:
for name, df in tables.items():
    print(f'\n{name}: head and tail')
    display(pd.concat([df.head(3),df.tail(3)]))


movies: head and tail


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000,262797249.0,17.015539,6.9,2413.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995
9079,162672,402672,tt3859980,Mohenjo Daro,Mohenjo Daro,hi,2016-08-11,155.0,15050000,16180000.0,1.423358,6.7,26.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 18, '...","[{'name': 'UTV Motion Pictures', 'id': 2320}, ...","[{'iso_3166_1': 'IN', 'name': 'India'}]",False,Released,Adventure|Drama|History|Romance,Adventure,2016
9080,163056,315011,tt4262980,Shin Godzilla,シン・ゴジラ,ja,2016-07-29,120.0,15000000,77000000.0,9.285519,6.6,152.0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'name': 'Cine Bazar', 'id': 5896}, {'name': ...","[{'iso_3166_1': 'JP', 'name': 'Japan'}]",False,Released,Action|Adventure|Drama|Horror|Science Fiction,Action,2016
9081,163949,391698,tt2531318,The Beatles: Eight Days a Week - The Touring Y...,The Beatles: Eight Days a Week - The Touring Y...,en,2016-09-15,99.0,0,0.0,7.078301,7.6,92.0,"[{'id': 99, 'name': 'Documentary'}, {'id': 104...","[{'name': 'Imagine Entertainment', 'id': 23}, ...","[{'iso_3166_1': 'GB', 'name': 'United Kingdom'...",False,Released,Documentary|Music,Documentary,2016



ratings: head and tail


,userId,movieId,rating,timestamp,rating_datetime
0,1,31,2.5,1260759144,2009-12-14 02:52:24
1,1,1029,3.0,1260759179,2009-12-14 02:52:59
2,1,1061,3.0,1260759182,2009-12-14 02:53:02
99807,671,6365,4.0,1070940363,2003-12-09 03:26:03
99808,671,6385,2.5,1070979663,2003-12-09 14:21:03
99809,671,6565,3.5,1074784724,2004-01-22 15:18:44



links: head and tail


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
9122,163949,2531318,391698.0
9123,164977,27660,137608.0
9124,164979,3447228,410803.0



credits: head and tail


,tmdbId,director,top_cast
0,862,John Lasseter,Tom Hanks|Tim Allen|Don Rickles
1,8844,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst
2,15602,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret
9079,402672,Ashutosh Gowariker,Hrithik Roshan|Pooja Hegde|Kabir Bedi
9080,315011,Hideaki Anno,Hiroki Hasegawa|Yutaka Takenouchi|Satomi Ishihara
9081,391698,Ron Howard,Paul McCartney|Ringo Starr|John Lennon



keywords: head and tail


,tmdbId,keywords_list
0,862,jealousy|toy|boy|friendship|friends|rivalry|bo...
1,8844,board game|disappearance|based on children's b...
2,15602,fishing|best friend|duringcreditsstinger|old men
9079,402672,bollywood
9080,315011,monster|godzilla|giant monster|destruction|kai...
9081,391698,music|documentary


### Head & Tail Results
The head and tail output resulted in some key findings:  
    - genres, production_companies, and production_countries are lists.  
    - genre list is genres reformatted as a string list delimited by "|".  
    - I'm not sure yet how ratings[timestamp] will be used. It may be redundant for this analysis.  
    - top_cast is a string list delimited by "|".  
    - keywords_list appears very detailed. Also a string seperated by "|".  

---

Now that we've reviewed the head and tail. We'll loop through the describe function.

In [5]:
# For every name in tables dictionary print a title and the describe results
for name, df in tables.items():
    print((f'{name}: Describe Summary'))
    display(df.describe().T)

#movies.sort_values(by= 'runtime', ascending= False)

movies: Describe Summary


,count,mean,std,min,25%,50%,75%,max
movieId,9082.0,3.096374e+04,4.065969e+04,1.000000,2843.250000,6265.500000,5.602625e+04,1.639490e+05
tmdbId,9082.0,3.871046e+04,6.212341e+04,2.000000,9446.250000,15775.000000,3.901025e+04,4.164370e+05
runtime,9082.0,1.056614e+02,3.035071e+01,0.000000,93.000000,102.000000,1.150000e+02,1.140000e+03
budget,9082.0,1.664789e+07,3.345276e+07,0.000000,0.000000,100000.000000,2.000000e+07,3.800000e+08
revenue,9082.0,4.918728e+07,1.301801e+08,0.000000,0.000000,271608.000000,3.672528e+07,2.787965e+09
popularity,9082.0,7.268905e+00,9.372314e+00,0.000004,2.954521,6.418871,9.861742e+00,5.474883e+02
vote_average,9082.0,6.362178e+00,1.038748e+00,0.000000,5.800000,6.500000,7.000000e+00,1.000000e+01
vote_count,9082.0,4.390509e+02,9.961565e+02,0.000000,29.000000,96.000000,3.657500e+02,1.407500e+04
release_year,9082.0,1.991952e+03,1.938771e+01,1902.000000,1984.000000,1997.000000,2.006000e+03,2.016000e+03


ratings: Describe Summary


,count,mean,min,25%,50%,75%,max,std
userId,99810.0,346.983228,1.0,182.0,367.0,520.0,671.0,195.166743
movieId,99810.0,12498.091013,1.0,1028.0,2406.0,5412.25,163949.0,26307.21951
rating,99810.0,3.542972,0.5,3.0,4.0,4.0,5.0,1.057939
timestamp,99810.0,1129347457.059673,789652009.0,965572660.75,1110395557.0,1296191573.75,1476640644.0,191574221.632708
rating_datetime,99810,2005-10-15 03:37:37.059673216,1995-01-09 11:46:49,2000-08-06 14:37:40.750000,2005-03-09 19:12:37,2011-01-28 05:12:53.750000128,2016-10-16 17:57:24,NaN


links: Describe Summary


,count,mean,std,min,25%,50%,75%,max
movieId,9125.0,31123.291836,40782.633604,1.0,2850.00,6290.0,56274.0,164979.0
imdbId,9125.0,479824.392329,743177.360844,417.0,88846.00,119778.0,428441.0,5794766.0
tmdbId,9112.0,39104.545544,62814.519801,2.0,9451.75,15852.0,39160.5,416437.0


credits: Describe Summary


,count,mean,std,min,25%,50%,75%,max
tmdbId,9082.0,38710.463885,62123.411079,2.0,9446.25,15775.0,39010.25,416437.0


keywords: Describe Summary


,count,mean,std,min,25%,50%,75%,max
tmdbId,9082.0,38710.463885,62123.411079,2.0,9446.25,15775.0,39010.25,416437.0


### Describe Function Results

The describe tables are easy to read and straight forward, but there are a few thing's I'd like to call out:

**movies**:   
    - Zero values exist in runtime, budget, and revenue. These are most likely missing values.  
    - There are extremely long runtimes. The max runtime is 1,140 mins. These will need to be cleaned from the data.  
    - Vote scoring system is from 1 to 10. The avg rating is 6.4.  
    - Min release year is 1902. Max year is 2016. Older movies may be outside the scope of this project.

**ratings**:  
    - The rating scale is from .5 to 5.
    - The reviews occured between 1995 and 2016.

***links, credits, and keywords data have no numeric data to describe.***   


---               
## Inspecting Missing, Duplicate, & Unusual Values

Next, we will profile each dataset and document issues that may require cleaning in the next project milestone. There are 5 sections, each dedicated to a dataset. 

For each dataset we will:  
    - Review data types.  
    - Check for missing values, empty strings, and zero values.  
    - Ensure no primary key is duplicated.  
    - Note suspicious, impossible, or confusing values


---
### Movies

The movies dataset contains over 9000 movies with 21 columns of details. We will profile the data below using python functions such as .dtypes, nunique, .isna.mean, and .isna.sum. It's more clear if we can view all the details regarding a dataset in a single table output.

In [6]:
# Create a dataframe to store movies data types and missing rate/count info.
# Target 'object' columns since that's how text/mixed data is stored.abs

# Convert all number columns into true or false so we can count where 0 is true
num_cols = movies.select_dtypes(include='number')
zero_mask = (num_cols == 0)

# Strip text columns nas,blanks to make sure they are really empty.
text_cols = movies.select_dtypes(include=['object'])
empty_mask = text_cols.apply(lambda col: col.dropna().astype(str).str.strip() == '')

# 3. Build the DataFrame
movies_profile = pd.DataFrame({
    'dtype': movies.dtypes, 
    'unique_values' : movies.nunique(),
    'duplicate_values' : movies.shape[0] - movies.nunique(),
    'missing_rate': round(movies.isna().mean() * 100,2), 
    'missing_count': movies.isna().sum(),
    'zero_count': zero_mask.sum().reindex(movies.columns, fill_value=0),
    'zero_rate': round(zero_mask.mean().reindex(movies.columns, fill_value=0) * 100,2),
    'empty_count': empty_mask.sum().reindex(movies.columns, fill_value=0),
    'empty_rate': round(empty_mask.mean().reindex(movies.columns, fill_value=0) * 100,2)
})

print("Movies Profile Table")
display(movies_profile.sort_values(by='missing_rate', ascending=False))


Movies Profile Table


,dtype,unique_values,duplicate_values,missing_rate,missing_count,zero_count,zero_rate,empty_count,empty_rate
genres_list,object,1694,7388,0.39,35,0,0.00,0,0.0
primary_genre,object,20,9062,0.39,35,0,0.00,0,0.0
status,object,4,9078,0.02,2,0,0.00,0,0.0
tmdbId,int64,9082,0,0.00,0,0,0.00,0,0
movieId,int64,9082,0,0.00,0,0,0.00,0,0
original_title,object,8841,241,0.00,0,0,0.00,0,0.0
title,object,8809,273,0.00,0,0,0.00,0,0.0
imdb_id,object,9082,0,0.00,0,0,0.00,0,0.0
original_language,object,42,9040,0.00,0,0,0.00,0,0.0
revenue,float64,4555,4527,0.00,0,4322,47.59,0,0


The output table generated above provides some key profiling information:

1) **dtypes** : There are a few things that could be changed.
    - ***release_date*** is an object. It should be converted to a date using pd.to_datetime().
    - ***vote_count*** is set as float64 but it's a whole number so it should be int64.
    - ***revenue*** and ***budget*** represent dollars but are different dtypes. Let's plan on standardizing them as float64. Float64 allows for NaN values, which will prevent them from being included in math calculations.
    - (***primary_genre, status, original_language***) contain few unqiue values. They could be converted to ***category*** data type. Doing so will improve grouping and sorting processing speeds.  

2) **Missing Values** : The output table display missing count/rate, zero count/rate, and empty count/rate.
    - Missing count/rate are calculated using isna mean and isna sum. Those functions identify null or missing values. The data appears to be extremely *complete* in regards to null and missing.
    - The zero count/rate reveal missing information that will impact the analysis. *Almost 50% of the revenue/budget values are 0*. This means when conducting analysis on revenue/budget we must omit these movieIds. Including them in any equations will skew the results.
    - empty count/rate shows 0 because there are no blank strings or values. There are only nulls/isna or 0.

3) **Duplicates** : The movies dataset has 9,082 rows.
    - ***movieId, tmdbId, and imdb_id*** have 9,082 rows. This makes logic sense and proves that the row level data represents a single movie.
    - It's possible to have duplicate movie ***titles***. The data spans from 1902 to 2016. Over almost 100 years there have been 273 movies with a title that's already been used.
    - Any column that represents a group or label is allowed to have duplicates. These are: ***grenes_list, primary_genre, status, original language, genres, adult, production countries, production companies, and release year***.
    - It's suspicious that ***revenue*** and ***budget*** have duplicates, but remember about 50% of them are 0. So 50% of them should be duplicates. Also consider that the dollars amounts have low granularity. The budget/revenue figures are a rough ball park of the real amounts. Which reduces the varitey of the data.
    - ***vote_average*** is based on a scale of 1-10. It makes sense that there are duplicates.

---
### Ratings

The ratings dataset is narrow. I don't expect the profiling process to be as in depth as movies.

In [7]:
# Create a dataframe to store ratings data types and missing rate/count info.
# Target 'object' columns since that's how text/mixed data is stored.abs

# Convert all number columns into true or false so we can count where 0 is true
num_cols = ratings.select_dtypes(include='number')
zero_mask = (num_cols == 0)

# Strip text columns nas,blanks to make sure they are really empty.
text_cols = ratings.select_dtypes(include=['object'])
empty_mask = text_cols.apply(lambda col: col.dropna().astype(str).str.strip() == '')

# 3. Build the DataFrame
ratings_profile = pd.DataFrame({
    'dtype': ratings.dtypes, 
    'unique_values' : ratings.nunique(),
    'duplicate_values' : ratings.shape[0] - ratings.nunique(),
    'missing_rate': round(ratings.isna().mean() * 100,2), 
    'missing_count': ratings.isna().sum(),
    'zero_count': zero_mask.sum().reindex(ratings.columns, fill_value=0),
    'zero_rate': round(zero_mask.mean().reindex(ratings.columns, fill_value=0) * 100,2),
    'empty_count': empty_mask.sum().reindex(ratings.columns, fill_value=0),
    'empty_rate': round(empty_mask.mean().reindex(ratings.columns, fill_value=0) * 100,2)
})

print("ratings Profile Table")
display(ratings_profile.sort_values(by='missing_rate', ascending=False))

ratings Profile Table


,dtype,unique_values,duplicate_values,missing_rate,missing_count,zero_count,zero_rate,empty_count,empty_rate
userId,int64,671,99139,0.0,0,0,0.0,0.0,0.0
movieId,int64,9025,90785,0.0,0,0,0.0,0.0,0.0
rating,float64,10,99800,0.0,0,0,0.0,0.0,0.0
timestamp,int64,77979,21831,0.0,0,0,0.0,0.0,0.0
rating_datetime,datetime64[ns],77979,21831,0.0,0,0,0.0,0.0,0.0


- One intesting thing to note in the output is there are only 671 unique userIds. It is possible that 671 users created 99,139 reviews. We will assume that's accurate and all reviews are legitimate.
- There are 10 unique values for rating. This is because the rating scale is in increments of .5 from .5 to 5.

---
### Links

The links dataset is narrow. I don't expect the profiling process to be as in depth as movies.

In [8]:
# Create a dataframe to store links data types and missing rate/count info.
# Target 'object' columns since that's how text/mixed data is stored.abs

# Convert all number columns into true or false so we can count where 0 is true
num_cols = links.select_dtypes(include='number')
zero_mask = (num_cols == 0)

# Strip text columns nas,blanks to make sure they are really empty.
text_cols = links.select_dtypes(include=['object'])
empty_mask = text_cols.apply(lambda col: col.dropna().astype(str).str.strip() == '')

# 3. Build the DataFrame
links_profile = pd.DataFrame({
    'dtype': links.dtypes, 
    'unique_values' : links.nunique(),
    'duplicate_values' : links.shape[0] - links.nunique(),
    'missing_rate': round(links.isna().mean() * 100,2), 
    'missing_count': links.isna().sum(),
    'zero_count': zero_mask.sum().reindex(links.columns, fill_value=0),
    'zero_rate': round(zero_mask.mean().reindex(links.columns, fill_value=0) * 100,2),
    'empty_count': empty_mask.sum().reindex(links.columns, fill_value=0),
    'empty_rate': round(empty_mask.mean().reindex(links.columns, fill_value=0) * 100,2)
})

print("links Profile Table")
display(links_profile.sort_values(by='missing_rate', ascending=False))

links Profile Table


,dtype,unique_values,duplicate_values,missing_rate,missing_count,zero_count,zero_rate,empty_count,empty_rate
tmdbId,float64,9112,13,0.14,13,0,0.0,0.0,0.0
movieId,int64,9125,0,0.00,0,0,0.0,0.0,0.0
imdbId,int64,9125,0,0.00,0,0,0.0,0.0,0.0


- The only thing worth noting is there are 13 missing the movie database ids. We wont be able to cross reference those 13 movies using the tmdbld since they don't exist.

---
### Credits

The credits dataset is narrow. I don't expect the profiling process to be as in depth as movies.

In [9]:
# Create a dataframe to store credits data types and missing rate/count info.
# Target 'object' columns since that's how text/mixed data is stored.abs

# Convert all number columns into true or false so we can count where 0 is true
num_cols = credits.select_dtypes(include='number')
zero_mask = (num_cols == 0)

# Strip text columns nas,blanks to make sure they are really empty.
text_cols = credits.select_dtypes(include=['object'])
empty_mask = text_cols.apply(lambda col: col.dropna().astype(str).str.strip() == '')

# 3. Build the DataFrame
credits_profile = pd.DataFrame({
    'dtype': credits.dtypes, 
    'unique_values' : credits.nunique(),
    'duplicate_values' : credits.shape[0] - credits.nunique(),
    'missing_rate': round(credits.isna().mean() * 100,2), 
    'missing_count': credits.isna().sum(),
    'zero_count': zero_mask.sum().reindex(credits.columns, fill_value=0),
    'zero_rate': round(zero_mask.mean().reindex(credits.columns, fill_value=0) * 100,2),
    'empty_count': empty_mask.sum().reindex(credits.columns, fill_value=0),
    'empty_rate': round(empty_mask.mean().reindex(credits.columns, fill_value=0) * 100,2)
})

print("credits Profile Table")
display(credits_profile.sort_values(by='missing_rate', ascending=False))

credits Profile Table


,dtype,unique_values,duplicate_values,missing_rate,missing_count,zero_count,zero_rate,empty_count,empty_rate
top_cast,object,8837,245,0.99,90,0,0.0,0,0.0
director,object,3601,5481,0.26,24,0,0.0,0,0.0
tmdbId,int64,9082,0,0.00,0,0,0.0,0,0


In [10]:
# Return the top 10 most frequent values in the director and top_cast columns
display(credits['director'].value_counts().head(10))
display(credits['top_cast'].value_counts().head(10))

director
Woody Allen          42
Alfred Hitchcock     38
Clint Eastwood       30
Steven Spielberg     30
Martin Scorsese      29
Steven Soderbergh    25
Sidney Lumet         25
Robert Altman        23
Werner Herzog        23
Ron Howard           22
Name: count, dtype: int64

top_cast
Daniel Radcliffe|Rupert Grint|Emma Watson           7
William Shatner|Leonard Nimoy|DeForest Kelley       6
Groucho Marx|Chico Marx|Harpo Marx                  6
Louis C.K.                                          6
Groucho Marx|Harpo Marx|Chico Marx                  5
Mike Myers|Eddie Murphy|Cameron Diaz                5
George Carlin                                       5
Sylvester Stallone|Talia Shire|Burt Young           5
Patrick Stewart|Jonathan Frakes|Brent Spiner        4
Jennifer Lawrence|Josh Hutcherson|Liam Hemsworth    4
Name: count, dtype: int64

The missing rate is insignificant in the credits table. I pulled the value_counts to review the top duplicates from the director and top_casts tables.

We can see that the directors and actors can have dozens of movies associated with them.

---
### Keywords

The keywords dataset is narrow. I don't expect the profiling process to be as in depth as movies.

In [11]:
# Create a dataframe to store keywords data types and missing rate/count info.
# Target 'object' columns since that's how text/mixed data is stored.abs

# Convert all number columns into true or false so we can count where 0 is true
num_cols = keywords.select_dtypes(include='number')
zero_mask = (num_cols == 0)

# Strip text columns nas,blanks to make sure they are really empty.
text_cols = keywords.select_dtypes(include=['object'])
empty_mask = text_cols.apply(lambda col: col.dropna().astype(str).str.strip() == '')

# 3. Build the DataFrame
keywords_profile = pd.DataFrame({
    'dtype': keywords.dtypes, 
    'unique_values' : keywords.nunique(),
    'duplicate_values' : keywords.shape[0] - keywords.nunique(),
    'missing_rate': round(keywords.isna().mean() * 100,2), 
    'missing_count': keywords.isna().sum(),
    'zero_count': zero_mask.sum().reindex(keywords.columns, fill_value=0),
    'zero_rate': round(zero_mask.mean().reindex(keywords.columns, fill_value=0) * 100,2),
    'empty_count': empty_mask.sum().reindex(keywords.columns, fill_value=0),
    'empty_rate': round(empty_mask.mean().reindex(keywords.columns, fill_value=0) * 100,2)
})

print("keywords Profile Table")
display(keywords_profile.sort_values(by='missing_rate', ascending=False))

keywords Profile Table


,dtype,unique_values,duplicate_values,missing_rate,missing_count,zero_count,zero_rate,empty_count,empty_rate
keywords_list,object,7951,1131,8.38,761,0,0.0,0,0.0
tmdbId,int64,9082,0,0.00,0,0,0.0,0,0.0


The missing rate is insignificant is about 8% in the keywords table.

---               
## Initial Cleaning Plan

The initial cleaning plan will consist of the following steps:
1) Create copies of all raw data tables. All transformation will be done on copies. We don't want to change the raw data.
2) Log or create tables of any movieIds with missing or zero data.
3) Convert release_date to a date using pd.to_datetime(). Use errors = 'coerce' and log NaT.
4) Replace budget and revenue 0s with NaN.
5) Standardize the revenue and budget formats. Make them both float64.
6) Convert primary_genre, status, and original_language to category dtypes.
7) Convert vote_count, vote_average, and runtime 0 values to NaN.
8) Convert vote_count from float64 to Int64. It's a whole number.
9) Drop movies[adult]. All values are false. It provides no value to this analysis.
10) Drop movies that are missing all of the following: ratings in either table, genre attributes, or revenue. There are very few but best to remove them.
11) All movies dropped will be documented and placed in dropped dataset/list.
12) Parse through any lists included in the data sets and make a separate table (ex: movie_genres_parsed, keywords_parsed, cast_parsed)
13) Create a movies_ROI dataset that has movies with both revenue and budget.


---               
## Potential target or outcome variable

*see "The Model" section at the top of the notebook.*


---               
## Early ethics, privacy, governance, or responsible-use concerns


In [ ]:
#Random code used for validation and queries.

display(movies.query("revenue > 0 and release_year < 1995"))

print(ratings.columns)
display(
    ratings[ratings['movieId'].isin(movies.query('release_year < 1995')['movieId'])]
    .merge(movies[['movieId', 'title', 'release_date']], on='movieId')
    [['title', 'release_date', 'rating_datetime']]
    .head(20)
)

,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
101,111,103,tt0075314,Taxi Driver,Taxi Driver,en,1976-02-07,114.0,1300000,28262574.0,14.092713,8.1,2632.0,"[{'id': 80, 'name': 'Crime'}, {'id': 18, 'name...","[{'name': 'Columbia Pictures Corporation', 'id...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Crime|Drama,Crime,1976
110,122,11066,tt0103859,Boomerang,Boomerang,en,1992-06-30,117.0,40000000,70100000.0,6.506983,5.4,139.0,"[{'id': 35, 'name': 'Comedy'}, {'id': 10749, '...","[{'name': 'Paramount Pictures', 'id': 4}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy|Romance,Comedy,1992
141,162,26564,tt0109508,Crumb,Crumb,en,1994-09-10,119.0,0,3174695.0,3.620388,7.6,71.0,"[{'id': 99, 'name': 'Documentary'}]","[{'name': 'Superior Pictures', 'id': 4532}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Documentary,Documentary,1994
187,213,50797,tt0111579,Burnt by the Sun,Утомлённые солнцем,ru,1994-05-01,135.0,2800000,2616503.0,2.486073,6.7,35.0,"[{'id': 18, 'name': 'Drama'}]","[{'name': 'Canal+', 'id': 5358}, {'name': 'Stu...","[{'iso_3166_1': 'FR', 'name': 'France'}, {'iso...",False,Released,Drama,Drama,1994
188,214,19155,tt0110882,Before the Rain,Before the Rain,en,1994-06-09,113.0,0,763847.0,3.525649,7.7,45.0,"[{'id': 18, 'name': 'Drama'}, {'id': 10752, 'n...","[{'name': 'PolyGram Audiovisuel', 'id': 6367},...","[{'iso_3166_1': 'GB', 'name': 'United Kingdom'...",False,Released,Drama|War,Drama,1994
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,62383,30266,tt0006333,"20,000 Leagues Under the Sea","20,000 Leagues Under the Sea",en,1916-12-24,105.0,200000,8000000.0,0.917777,6.0,7.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 18, '...","[{'name': 'The Universal Film Mfg. Co.', 'id':...","[{'iso_3166_1': 'BS', 'name': 'Bahamas'}, {'is...",False,Released,Adventure|Drama|Action|Science Fiction,Adventure,1916
7162,66915,20421,tt0102802,Rock-A-Doodle,Rock-A-Doodle,en,1991-08-02,77.0,18000000,11657385.0,3.602031,5.3,46.0,"[{'id': 35, 'name': 'Comedy'}, {'id': 10751, '...","[{'name': 'Goldcrest Films International', 'id...","[{'iso_3166_1': 'IE', 'name': 'Ireland'}, {'is...",False,Released,Comedy|Family|Animation|Fantasy|Music,Comedy,1991
7256,69945,20174,tt0046969,The Fast and the Furious,The Fast and the Furious,en,1955-02-15,73.0,66000,250000.0,1.114697,3.8,6.0,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam...","[{'name': 'Palo Alto Productions', 'id': 3733}...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Action|Crime|Drama,Action,1955
7263,70201,47870,tt0095302,High Hopes,High Hopes,en,1988-09-01,112.0,0,1192322.0,1.369216,7.5,22.0,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",[],"[{'iso_3166_1': 'GB', 'name': 'United Kingdom'}]",False,Released,Comedy|Drama,Comedy,1988


Index(['userId', 'movieId', 'rating', 'timestamp', 'rating_datetime'], dtype='object')


,title,release_date,rating_datetime
0,Dumbo,1941-10-22,2009-12-14 02:52:59
1,Escape from New York,1981-05-22,2009-12-14 02:53:05
2,Cinema Paradiso,1988-11-17,2009-12-14 02:53:25
3,The Deer Hunter,1978-12-08,2009-12-14 02:52:31
4,Ben-Hur,1959-12-26,2009-12-14 02:53:07
5,Gandhi,1982-11-30,2009-12-14 02:52:28
6,Dracula,1992-11-13,2009-12-14 02:52:05
7,Cape Fear,1991-11-15,2009-12-14 02:52:11
8,Star Trek: The Motion Picture,1979-12-06,2009-12-14 02:52:15
9,The French Connection,1971-10-09,2009-12-14 02:53:11
